# MC8 — calibração do protocolo sob efeitos conhecidos

Este é o experimento que o parecer considera decisivo, e com razão: a contribuição do artigo é o
protocolo, então o protocolo precisa ser validado, não apenas aplicado.

A pergunta é se o procedimento completo, com seleção de features, imputação, escala e escolha de
bandwidth dentro dele, mantém a taxa nominal de erro tipo I e tem poder conhecido quando existe
efeito de magnitude controlada.

**Desenho.** Os rótulos vêm de uma mistura com proporção `p` entre duas regras: uma separável pelo
feature map quântico e outra separável por um kernel gaussiano clássico. Em `p = 0` não há nada que
o lado quântico capture e o clássico não capture, de modo que a vantagem verdadeira é nula e a
fração de rejeições estima a taxa de falso positivo. À medida que `p` cresce, a vantagem verdadeira
cresce com ela e a fração de rejeições traça a curva de poder.

**Regra de decisão.** Para cada conjunto simulado o piso é estimado por permutação dentro daquele
conjunto, e declara-se vantagem quando a estatística observada supera o percentil 95 do nulo. É
exatamente a regra usada nos corpora reais, com o mesmo pipeline, e é ela que está sendo auditada
aqui — não o estimador isolado.

In [ ]:
#@title Infraestrutura (painéis, kernels, protocolo) { display-mode: "form" }
import numpy as np, pandas as pd, time, itertools
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.kernel_approximation import RBFSampler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

def painel(titulo, linhas=(), tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            if ent:
                for i in range(q-1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T)**2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :]-A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K-np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x))

def ic95(v):
    v = np.asarray(v, float); n = len(v)
    if n < 2: return float(v.mean()), np.nan, np.nan
    se = v.std(ddof=1)/np.sqrt(n); tc = tdist.ppf(.975, n-1)
    return float(v.mean()), float(v.mean()-tc*se), float(v.mean()+tc*se)


In [ ]:
#@title Geração com efeito controlado { display-mode: "form" }
def _observavel(dim, rng):
    V = np.linalg.qr(rng.normal(size=(2**dim,)*2) + 1j*rng.normal(size=(2**dim,)*2))[0]
    D = np.diag([1.]*(2**(dim-1)) + [-1.]*(2**(dim-1)))
    return V.conj().T @ D @ V

def gera(n, dim, p, prevalencia=0.5, seed=0):
    """p = fração de rótulos ditados pela regra quântica; o resto segue uma regra gaussiana."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(0, np.pi, (n, dim))
    S = states(X, dim, 2, True)
    O = _observavel(dim, rng)
    v_q = np.einsum("ni,ij,nj->n", S.conj(), O, S).real
    centro = rng.uniform(0, np.pi, dim)
    v_c = -((X - centro)**2).sum(1)
    z = lambda v: (v - v.mean())/(v.std() + 1e-12)
    escore = p*z(v_q) + (1-p)*z(v_c)
    corte = np.quantile(escore, 1 - prevalencia)
    y = (escore > corte).astype(int)
    return X, y

painel("gerador",
       ["p = 0 &rarr; rótulo puramente clássico, vantagem quântica verdadeira nula",
        "p = 1 &rarr; rótulo puramente quântico, vantagem máxima",
        "prevalência controla o desbalanceamento das classes"],
       destaque="Em p=0 o alvo é separável por uma gaussiana centrada em um ponto aleatório, "
                "portanto o braço clássico tem como acertá-lo. Esse é o nulo correto para a "
                "pergunta do artigo, que é sobre vantagem relativa e não sobre sinal nos dados.")

In [ ]:
#@title Estatística e regra de decisão { display-mode: "form" }
Q_SIM = 4                                #@param {type:"integer"}
SCALES_SIM = np.round(np.geomspace(0.05, 4.0, 10), 4)
FOLDS_SIM = 5                            #@param {type:"integer"}
N_PERM = 200                             #@param {type:"integer"}
ALFA = 0.05                              #@param {type:"number"}

def estatistica(X, y, folds=FOLDS_SIM, q=Q_SIM, seed=7):
    """Vantagem do kernel quântico sobre o surrogate, bandwidth por leave-one-fold-out."""
    reg, sk = {}, StratifiedKFold(folds, shuffle=True, random_state=seed)
    partes = list(sk.split(X, y))
    for f, (tr, te) in enumerate(partes):
        imp = SimpleImputer(strategy="median").fit(X[tr])
        Xtr, Xte = imp.transform(X[tr]), imp.transform(X[te])
        k = min(q, Xtr.shape[1])
        sel = SelectKBest(f_classif, k=k).fit(Xtr, y[tr])
        sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
        A = sc.transform(sel.transform(Xtr))
        B = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
        for cv in SCALES_SIM:
            Ac, Bc = A*cv, B*cv
            S1, S2 = states(Ac, k, 2, True), states(Bc, k, 2, True)
            K = gram_fid(S1, S1)
            m = SVC(kernel="precomputed").fit(K, y[tr])
            aq = roc_auc_score(y[te], m.decision_function(gram_fid(S2, S1)))
            gs = fit_surrogate(K, Ac)
            ms = SVC(kernel="rbf", gamma=gs).fit(Ac, y[tr])
            reg[(f, cv)] = (aq, roc_auc_score(y[te], ms.decision_function(Bc)))
    v = []
    for f in range(len(partes)):
        fora = {cv: np.mean([reg[(j, cv)][0] for j in range(len(partes)) if j != f])
                for cv in SCALES_SIM}
        c_star = max(fora, key=fora.get)
        aq, asr = reg[(f, c_star)]
        v.append(aq - asr)
    return float(np.mean(v))

def piso_permutacao(X, y, n_perm=N_PERM, seed=0):
    rng = np.random.default_rng(seed)
    nulo = np.array([estatistica(X, rng.permutation(y)) for _ in range(n_perm)])
    return float(np.quantile(nulo, 1-ALFA)), nulo

painel("regra de decisão",
       [f"estatística: vantagem média por fold, bandwidth escolhida fora do fold avaliado",
        f"piso: percentil {100*(1-ALFA):.0f} de {N_PERM} permutações dentro do próprio conjunto",
        "declara-se vantagem quando a estatística observada supera o piso"],
       destaque="É a mesma regra aplicada aos corpora reais. O que se mede aqui é se ela "
                "mantém a taxa nominal quando não há efeito, e que poder tem quando há.")

In [ ]:
#@title Grade de simulação e orçamento { display-mode: "form" }
P_EFEITO   = [0.0, 0.15, 0.30, 0.50, 0.80]
N_AMOSTRAS = [120, 240, 480]
PREVALENCIA = [0.50, 0.25]
R_REPLICAS = 40                          #@param {type:"integer"}

celulas = len(P_EFEITO)*len(N_AMOSTRAS)*len(PREVALENCIA)*R_REPLICAS
chamadas = celulas * (1 + N_PERM)
painel("orçamento da simulação", [
    f"{len(P_EFEITO)} efeitos &times; {len(N_AMOSTRAS)} tamanhos &times; "
    f"{len(PREVALENCIA)} prevalências &times; {R_REPLICAS} réplicas = <b>{celulas}</b> conjuntos",
    f"cada conjunto: 1 estatística + {N_PERM} permutações = {1+N_PERM} avaliações do pipeline",
    f"total &asymp; <b>{chamadas:,}</b> avaliações do pipeline completo",
    f"precisão das taxas: com {R_REPLICAS} réplicas, o IC95 de uma taxa de {ALFA:.2f} tem "
    f"semilargura de &plusmn;{1.96*np.sqrt(ALFA*(1-ALFA)/R_REPLICAS):.3f}"],
    destaque="Este é o experimento caro do artigo. Nada é reduzido automaticamente. "
             "Se precisar encurtar, <b>R_REPLICAS</b> controla a precisão das taxas e "
             "<b>N_PERM</b> controla a precisão do piso; os dois aparecem no relatório final "
             "com seu próprio erro, para que a redução fique visível em vez de escondida. "
             "Para separar uma taxa observada de 0,05 de uma de 0,10 seriam necessárias algumas "
             "centenas de réplicas por célula; com poucas dezenas a tabela de erro tipo I "
             "descreve a tendência, não decide a calibração.")

In [ ]:
#@title Executar { display-mode: "form" }
linhas, t0 = [], time.time()
for p, n, prev in itertools.product(P_EFEITO, N_AMOSTRAS, PREVALENCIA):
    for r in range(R_REPLICAS):
        X, y = gera(n, Q_SIM, p, prev, seed=10_000*int(p*100) + 100*n + r)
        if len(np.unique(y)) < 2: continue
        T = estatistica(X, y, seed=r)
        piso, nulo = piso_permutacao(X, y, seed=r)
        linhas.append(dict(p=p, n=n, prevalencia=prev, replica=r, T=T, piso=piso,
                           rejeita=bool(T > piso), nulo_sd=float(nulo.std())))
    print(f"p={p} n={n} prev={prev} — {time.time()-t0:.0f}s")

sim = pd.DataFrame(linhas)
sim.to_csv("mc8_calibracao.csv", index=False)
painel("simulação concluída", [f"{len(sim)} conjuntos em {time.time()-t0:.0f}s",
                               "bruto salvo em mc8_calibracao.csv"])

In [ ]:
#@title Taxa de erro tipo I observada contra a nominal { display-mode: "form" }
nulo = sim[sim.p == 0.0]
lin = []
for (n, prev), g in nulo.groupby(["n", "prevalencia"]):
    k, m = int(g.rejeita.sum()), len(g)
    taxa = k/m
    se = np.sqrt(taxa*(1-taxa)/m)
    lin.append(dict(n=n, prevalencia=prev, replicas=m, rejeicoes=k,
                    taxa_observada=round(taxa, 3),
                    ic95=f"[{max(0,taxa-1.96*se):.3f}, {min(1,taxa+1.96*se):.3f}]",
                    nominal=ALFA, calibrado=bool(abs(taxa-ALFA) <= 1.96*se)))
t1 = pd.DataFrame(lin)
ok = bool(t1.calibrado.all())
painel("erro tipo I",
       [f"nominal = {ALFA}", f"réplicas por célula = {R_REPLICAS}"],
       tabela=t1,
       destaque=("O procedimento mantém a taxa nominal em todas as células. É isso que autoriza "
                 "ler um resultado abaixo do piso como não rejeição, e não como simples ausência "
                 "de poder." if ok else
                 "<b>Alguma célula está fora da taxa nominal.</b> Enquanto isso não for corrigido, "
                 "nenhuma afirmação de ausência de vantagem no artigo está calibrada. Verifique "
                 "primeiro as células com n menor, onde a seleção de hiperparâmetros pesa mais."))

In [ ]:
#@title Curvas de poder { display-mode: "form" }
lin = []
for (n, prev, p), g in sim.groupby(["n", "prevalencia", "p"]):
    k, m = int(g.rejeita.sum()), len(g)
    taxa = k/m; se = np.sqrt(taxa*(1-taxa)/m)
    lin.append(dict(n=n, prevalencia=prev, p=p, poder=round(taxa, 3),
                    ic95=f"[{max(0,taxa-1.96*se):.3f}, {min(1,taxa+1.96*se):.3f}]",
                    T_mediano=round(float(g["T"].median()), 4),
                    piso_mediano=round(float(g.piso.median()), 4)))
t2 = pd.DataFrame(lin).sort_values(["prevalencia", "n", "p"])

# menor efeito com poder >= 80% por configuração
det = []
for (n, prev), g in t2.groupby(["n", "prevalencia"]):
    g2 = g[(g.p > 0) & (g.poder >= 0.8)]
    det.append(dict(n=n, prevalencia=prev,
                    menor_p_com_poder_80=(g2.p.min() if len(g2) else np.nan),
                    T_nesse_ponto=(float(g2.sort_values("p").T_mediano.iloc[0]) if len(g2) else np.nan)))
det = pd.DataFrame(det)

painel("poder por efeito, tamanho e prevalência",
       ["poder = fração de réplicas em que a estatística supera o piso",
        "T_mediano = vantagem observada; piso_mediano = valor crítico estimado"],
       tabela=t2.reset_index(drop=True))
painel("menor efeito detectável com poder de 80%", tabela=det,
       destaque="Este é o número que falta no manuscrito. Com ele, a frase sobre ausência de "
                "vantagem passa a ter um alcance declarado: o desenho enxergaria um efeito desta "
                "magnitude em 80% das vezes, e os efeitos observados nos corpora reais ficam "
                "abaixo disso.")

## O que isso resolve no parecer

O MC1 pede que o limiar seja demonstrado sob efeitos conhecidos e que a regra preserve suas
propriedades quando há seleção de hiperparâmetros: a tabela de erro tipo I responde exatamente isso,
porque a seleção de bandwidth está dentro do pipeline permutado. O MC8 pede taxas nominais contra
observadas e curvas de poder, que são as duas tabelas finais. E a última tabela fornece o número
que o artigo não tem hoje, o menor efeito detectável com poder de 80%, que é o que transforma a
ausência de vantagem em uma afirmação com alcance declarado.

Se alguma célula de erro tipo I sair fora da taxa nominal, isso é resultado e vai para o texto. Uma
regra de decisão que não se sustenta sob o próprio nulo é um achado sobre a prática de comparação
em QML, não um defeito a esconder.